In [ ]:
# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
from pathlib import Path

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import pingouin as pg

# 머신러닝 
from sklearn.model_selection import GroupShuffleSplit

from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('olist_project 루트를 찾을 수 없습니다. 노트북을 olist_project 또는 notebooks 폴더 안에서 실행하세요.')

PROJECT_ROOT = find_project_root()
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
CHART_DIR = OUTPUT_DIR / 'charts'
TABLE_DIR = OUTPUT_DIR / 'tables'
OUTPUT_DIR.mkdir(exist_ok=True)
CHART_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)


In [ ]:
DATA_PATH = DATA_PROCESSED_DIR / 'merged_final_data.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'{DATA_PATH} 파일이 없습니다. '
        'notebooks/01_preprocessing.ipynb를 처음부터 끝까지 실행해 최종 분석용 CSV를 먼저 생성하세요.'
    )

df = pd.read_csv(DATA_PATH)
df2 = df.copy()


In [ ]:
cols = df2.columns
cols


In [ ]:
# train / test split
# EDA도 모델링과 동일하게 order_id 기준 그룹 분리를 사용해 한 주문의 여러 아이템이 서로 다른 split에 섞이지 않도록 합니다.

X = df2[cols]
y = df2['review_score']
groups = df2['order_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_valid = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train_valid = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print('order_id overlap:', len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print('train_valid shape:', X_train_valid.shape)
print('test shape:', X_test.shape)
print('review_score ratio by split')
print(pd.DataFrame({
    'train_valid': y_train_valid.value_counts(normalize=True).sort_index(),
    'test': y_test.value_counts(normalize=True).sort_index(),
}).round(4))


### 본격적인 EDA 시작


In [ ]:
print(f"test 셋 분리 전 df : {len(df2)}")
df2 = X_train_valid.copy()
print(f"test 셋 분리 후 df : {len(df2)}")


In [ ]:
df2.to_csv(DATA_PROCESSED_DIR / 'merged_train_data.csv', index=False)


In [ ]:
df2['review_score'].value_counts()


In [ ]:
plt.figure(figsize=(12, 8))

# 각 점수(1~5)에 매칭될 색상 딕셔너리 정의
# 낮은 점수는 오렌지 계열(#DD8452), 높은 점수는 블루 계열(#4C72B0)
score_palette = {
    1: "#DD8452", # 매우 낮음 (오렌지)
    2: "#E1A679", # 낮음 (연한 오렌지)
    3: "#B9C0C9", # 보통 (회색/중립)
    4: "#7091C2", # 높음 (연한 블루)
    5: "#4C72B0"  # 매우 높음 (딥 블루)
}

# 그래프 그리기
sns.countplot(
    data=df2, 
    x='review_score', 
    palette=score_palette, 
    hue='review_score',
    legend=False,   
    edgecolor='gray',
    alpha=0.8   
)

plt.title('리뷰 스코어(1~5) 분포 현황', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Review Score (점수)', fontsize=12)
plt.ylabel('데이터 개수 (건수)', fontsize=12)

# 각 막대 위에 실제 빈도수(숫자) 표시
ax = plt.gca()
for p in ax.patches:
    height = p.get_height()
    if height > 0: # 데이터가 있는 경우만 표시
        ax.annotate(f'{int(height):,}', 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha = 'center', va = 'center', 
                    xytext = (0, 10), 
                    textcoords = 'offset points',
                    fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))

# 데이터 필터링 (상위 5% 극단치 제외)
delivery_data = df2[df2['delivery_days'] < df2['delivery_days'].quantile(0.95)]['delivery_days']

# 히스토그램 그리기
sns.histplot(
    delivery_data, 
    bins=25, 
    color='#4C72B0',    
    kde=True,            # 밀도 곡선 추가
    line_kws={'linewidth': 3}, 
    edgecolor='white',   # 막대 사이 구분선 추가
    alpha=0.7       
)

# 세부 디자인 설정
plt.title('전체 배송 소요일 분포 (상위 5% 극단치 제외)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('배송 소요일 (Delivery Days)', fontsize=12)
plt.ylabel('빈도 (Frequency)', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# --- 시각화 ---
plt.figure(figsize=(12, 8))

# 데이터 필터링: 지연된 건(양수) 중 상위 5% 극단치 제외
delayed_df = df2[df2['delay_days'] > 0]
plot_data = delayed_df[delayed_df['delay_days'] < delayed_df['delay_days'].quantile(0.95)]['delay_days']

# 히스토그램 그리기
sns.histplot(
    plot_data, 
    bins=25, 
    color='#4C72B0',   
    kde=True,   
    line_kws={'linewidth': 3}, 
    edgecolor='white', 
    alpha=0.7
)

# 세부 디자인 설정
plt.title('배송 지연 일수 분포 (지연 발생 건 한정, 상위 5% 제외)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('지연 일수 (Delay Days)', fontsize=12)
plt.ylabel('빈도 (Frequency)', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# delay X distance  -> review

delay_distance = df2.groupby(['delay_days_cat','distance_cat'])['review_score'].mean().unstack()

x_order = [
    'Urban/Last-Mile',
    'Short-Haul',
    'Mid-Haul',
    'Long-Haul',
    'Continental'
]

y_order = [
    '조기',
    '정시',
    '1-3일 지연',
    '4-6일 지연',
    '7-9일 지연',
    '10일 이상 지연'
]

delay_distance = delay_distance.loc[y_order, x_order]


plt.figure(figsize=(10,6))

sns.heatmap(
    delay_distance,
    annot=True,
    fmt=".2f",
    cmap='RdBu'
)

plt.title("Delay X Distance가 리뷰에 미치는 영향")
plt.show()


In [ ]:
# 주(State) 및 중분류(sub_category)별 주문 건수 집계
state_category_counts = df2.groupby(['customer_state', 'sub_category'])['order_id'].count().reset_index()
state_category_counts.rename(columns={'order_id': 'order_count'}, inplace=True)

# 각 주(State)별로 주문 건수가 많은 순서대로 순위(Rank) 매기기
state_category_counts['rank'] = state_category_counts.groupby('customer_state')['order_count'].rank(method='first', ascending=False)

# 각 주별 Top 3 카테고리만 추출
top3_categories = state_category_counts[state_category_counts['rank'] <= 3].copy()

# 보기 편하게 피벗 테이블로 변환 (State를 인덱스로, 1, 2, 3위를 컬럼으로 배치)
top3_categories['category_info'] = top3_categories['sub_category'] + " (" + top3_categories['order_count'].astype(int).astype(str) + "건)"

pivot_top3 = top3_categories.pivot(index='customer_state', columns='rank', values='category_info').reset_index()
pivot_top3.columns = ['고객 거주 주 (State)', '1위 카테고리', '2위 카테고리', '3위 카테고리']

# 전체 주문 건수가 많은 주(State) 순서대로 정렬하기 위해 총 주문 건수 계산 및 병합
state_total_orders = df2.groupby('customer_state')['order_id'].count().reset_index()
state_total_orders.rename(columns={'order_id': '총 주문 건수'}, inplace=True)

final_table = pd.merge(state_total_orders, pivot_top3, left_on='customer_state', right_on='고객 거주 주 (State)')
final_table = final_table.sort_values(by='총 주문 건수', ascending=False).reset_index(drop=True)

# 컬럼 순서 정리
final_table = final_table[['고객 거주 주 (State)', '총 주문 건수', '1위 카테고리', '2위 카테고리', '3위 카테고리']]

# 결과 출력 (전체 27개 주 데이터 모두 표시)
print("=== 지역(State)별 고객들이 가장 많이 구매한 Top 3 상품 카테고리 (중분류 적용) ===")
display(final_table)


In [ ]:
# 상위 5개 주(State) 추출 (SP부터 PE 지역까지)
top5_states = df2['customer_state'].value_counts().nlargest(5).index
df_top5 = df2[df2['customer_state'].isin(top5_states)].copy()

# 주(State)별, 중분류 카테고리(sub_category)별 주문 건수 집계
state_cat_counts = df_top5.groupby(['customer_state', 'sub_category']).size().reset_index(name='order_count')

# 주별로 주문 건수 내림차순 정렬 후 순위(Rank) 부여
state_cat_counts = state_cat_counts.sort_values(['customer_state', 'order_count'], ascending=[True, False])
state_cat_counts['rank'] = state_cat_counts.groupby('customer_state').cumcount() + 1

# 1~3위와 'etc'로 그룹화하는 함수
def get_rank_group(r):
    if r == 1: return '1위'
    elif r == 2: return '2위'
    elif r == 3: return '3위'
    else: return 'etc'

state_cat_counts['rank_group'] = state_cat_counts['rank'].apply(get_rank_group)

# 등수(rank_group)별로 데이터 병합
grouped = state_cat_counts.groupby(['customer_state', 'rank_group']).agg(
    order_count=('order_count', 'sum'),
    category_name=('sub_category', lambda x: x.iloc[0] if len(x) == 1 else 'etc')
).reset_index()

# 피벗 테이블 생성 및 100% 비율로 변환
pivot_counts = grouped.pivot(index='customer_state', columns='rank_group', values='order_count').fillna(0)
pivot_names = grouped.pivot(index='customer_state', columns='rank_group', values='category_name').fillna('')

# 각 행(State)의 합계로 나누어 비율(%) 계산
pivot_percentages = pivot_counts.div(pivot_counts.sum(axis=1), axis=0) * 100

# 컬럼(순위) 순서 고정
col_order = ['1위', '2위', '3위', 'etc']
pivot_percentages = pivot_percentages[col_order]
pivot_names = pivot_names[col_order]

# X축(State)을 총 주문 건수가 많은 순서대로 정렬
pivot_percentages = pivot_percentages.loc[top5_states]
pivot_names = pivot_names.loc[top5_states]

# 시각화 (100% 누적 막대 그래프)
fig, ax = plt.subplots(figsize=(12, 8))

# 1위(파랑), 2위(주황), 3위(초록), etc(회색) 색상 설정
colors = ['#4C72B0', '#DD8452', '#55A868', '#D3D3D3'] 

# 데이터가 막대 높이 모두 100으로
pivot_percentages.plot(kind='bar', stacked=True, color=colors, ax=ax, width=0.75, edgecolor='black', alpha=0.8, linewidth=0.5)

ax.set_title('상위 5개 State별 카테고리 비중', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('고객 거주 주 (Customer State) -> 총 주문 건수 내림차순', fontsize=14, fontweight='bold')
ax.set_ylabel('카테고리 비중 (%)', fontsize=14, fontweight='bold')
plt.xticks(rotation=0, fontsize=12)
ax.set_ylim(0, 100) # Y축을 0~100%로 고정

# x축 텍스트 매핑
state_korean_mapping = {
    'SP': '상파울루', 'RJ': '리우 데 자네이루', 'MG': '미나스 제라이스', 
    'RS': '리우 그란데 두 술', 'PR': '파라나'
}

new_labels = [state_korean_mapping.get(label.get_text(), label.get_text()) for label in ax.get_xticklabels()]
ax.set_xticklabels(new_labels, rotation=0, ha='center', fontsize=12)

# 모든 막대 중앙에 텍스트 달아주기
for i, state in enumerate(pivot_percentages.index):
    y_offset = 0 # 텍스트를 쓸 y축 (퍼센트) 시작 위치
    
    for col in col_order:
        val_pct = pivot_percentages.loc[state, col]
        cat_name = pivot_names.loc[state, col]
        
        if val_pct > 0:
            y_pos = y_offset + (val_pct / 2) # 막대의 정중앙 (퍼센트 기준)
            
            # 텍스트가 겹치지 않도록 비중이 3% 이상일 때만 텍스트 렌더링
            if val_pct > 3.0:
                if col == 'etc':
                    # 기타
                    label_text = f"etc\n({val_pct:.1f}%)"
                    text_color = '#333333'
                    fontsize = 9
                else:
                    # 1~3위는 카테고리명과 퍼센트 표시
                    label_text = f"{cat_name}\n({val_pct:.1f}%)"
                    text_color = 'black'
                    fontsize = 9
                    
                ax.text(i, y_pos, label_text, ha='center', va='center', color=text_color, fontsize=fontsize, fontweight='bold')
            
            y_offset += val_pct # 다음 막대를 위해 y 위치 업데이트

# 범례 설정
plt.legend(title='카테고리 순위', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# 월별, 요일별 주요 일수 추이 비교

# 2x4 서브플롯 생성
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

days = ['delivery_days','dispatch_days','approved_days']
colors = ["#4C72B0", "#DD8452", "#55A868"]

for i, day in enumerate(days):
    ax1 = axes[0][i]
    ax2 = axes[1][i]

    monthly = df2.groupby('order_purchase_month')[day].mean().reset_index()
    daily = df2.groupby('order_purchase_dayofweek')[day].mean().reset_index()

    # 월별 막대 그래프
    sns.barplot(data=monthly, x='order_purchase_month', y=day,
                ax=ax1, color=colors[i], alpha=0.8)
    
    # 요일별 막대 그래프
    sns.barplot(data=daily, x='order_purchase_dayofweek', y=day,
                ax=ax2, color=colors[i], alpha=0.8)
    
    # 타이틀 및 라벨 설정
    ax1.set_title(f"월별 {day} 평균")
    ax2.set_title(f"요일별 {day} 평균")
    for ax in [ax1, ax2]:
        ax.set_xlabel("") # x라벨 중복 제거
        ax.set_ylabel("일수(Days)")
        ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# 월별 거래량

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

monthly = df2.groupby('order_purchase_month').size().reset_index(name='count')
daily = df2.groupby('order_purchase_dayofweek').size().reset_index(name='count')

# 월별 막대 그래프
sns.barplot(data=monthly, x='order_purchase_month', y='count', 
            ax=axes[0], color="#4C72B0", alpha=0.8)
axes[0].set_title(f"월별 총 거래량")
    
# 요일별 막대 그래프
sns.barplot(data=daily, x='order_purchase_dayofweek', y='count',
                ax=axes[1], color="#DD8452", alpha=0.8)
axes[1].set_title(f"요일별 총 거래량")
    
for ax in [axes[0], axes[1]]:
    ax.set_xlabel("")
    ax.set_ylabel("거래량(count)")
    ax.tick_params(axis='x', rotation=45)
# 막대 상단에 숫자 표시 (선택 사항)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha = 'center', va = 'center', xytext = (0, 9), textcoords = 'offset points', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Figure(도화지)와 Axes(그래프 구역) 객체 명확히 생성
fig, ax = plt.subplots(figsize=(12, 8))

# 데이터 집계
monthly = df2.groupby('order_purchase_month').size().reset_index(name='count')

# 월별 막대 그래프 그리기 (ax=plt 대신 ax=ax 할당)
sns.barplot(data=monthly, x='order_purchase_month', y='count', 
            ax=ax, color="#4C72B0", alpha=0.8)

# 텍스트 및 축 설정 (plt.set_x가 아닌 ax.set_x 사용)
ax.set_title("월별 총 거래량", fontsize=16, pad=15)
ax.set_ylabel("거래량 (count)", fontsize=12)
ax.set_xlabel("주문 월 (Month)", fontsize=12)
ax.tick_params(axis='x', rotation=45)
    
# 막대 상단에 숫자 표시
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',  # :, 를 넣어서 천 단위 콤마 추가 (보기 좋게)
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 9), textcoords='offset points', fontsize=10)

plt.tight_layout()
plt.show()


Carnival (Feb 14-17), New Year's Day (Jan 1), Independence Day (Sept 7), and Christmas (Dec 25)


- 월별로 배송에 걸리는 일수 (delivery_days : Delivered - Purchase) 를 확인했을 때 1,2,3,11,12월에 배송소요일이 크게 나오고, 6,7,8월에 배송소요일이 적게 나온다
- 이는 계절성과 관련이 있지않을까?
- 월별 거래량을 확인했을 때에도 오히려 배송소요일이 크게 나온 달보다 배송소요일이 적게 나온 달의 월별 거래량이 크게 나왔기 때문에 거래량으로 인한 배송지연은 아니라고 보여진다
- 브라질은 가톨릭 신자가 많아 크리스마스에 성대한 기념 행사가 일어나는데 이와 관련하여 12월에 배송지연이 많이 발생하는것 아닐까?
- 또한 2월 즈음에도 '카니발'이라는 중요한 축제도 열리기 때문에 이러한 연휴가 주로 껴있는 연말,연초에 배송이 오래 걸리고 이는 결국 거래량과는 무관하다고 보여진다


In [ ]:
# 월별 리뷰스코어 변화 -- 선그래프 
plt.figure(figsize=(12, 8))
sns.lineplot(data=df2, x='order_purchase_month', y='review_score', 
                 color="#4C72B0", marker='o', markersize=3, errorbar=('ci', 95))
plt.title("월별 리뷰스코어 변화 -- 선그래프")
plt.tight_layout()
plt.show()


In [ ]:
# 월별 리뷰스코어 변화 -- 막대그래프 
plt.figure(figsize=(12, 8))
sns.barplot(data=df2, x='order_purchase_month', y='review_score', 
                 color="#4C72B0", alpha=0.8)
plt.title("월별 리뷰스코어 변화 -- 막대그래프", fontsize=20, pad=15)
plt.xlabel('')
plt.tight_layout()
plt.tick_params(axis='x', labelsize=15, rotation=45)
plt.show()


월별 delivery_days가 차이가 나서 이에 따른 월별 review_score 또한 특징적인 추이가 나타날까 궁금하여 lineplot과 barplot을 그려보았지만, 

사실상 눈에 띄는 변화는 보이지 않았음


In [ ]:
# 배송 소요 일수에 따른 리뷰 스코어 변화

# 1. 시각화 대상 컬럼과 수식 정의
plot_info = [
    {'col': 'delivery_days', 'label': 'Delivery Days\n(Delivered - Purchase)', 'color': '#4C72B0'},
    {'col': 'dispatch_days', 'label': 'Dispatch Days\n(Carrier - Purchase)', 'color': '#DD8452'},
    {'col': 'expected_delivery_days', 'label': 'Expected Days\n(Estimated - Purchase)', 'color': '#55A868'},
    {'col': 'delay_days_int', 'label': 'Delay Days\n(Delivered - Estimated)', 'color': '#C44E52'}
]

# 2. 2x2 서브플롯 생성
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, info in enumerate(plot_info):
    col = info['col']
   
    # 너무 적은 샘플(롱테일)로 인한 왜곡 방지를 위해 상위 95% 구간까지만 시각화 (이상치 제외)
    upper_limit = df2[col].quantile(0.95)
    lower_limit = df2[col].quantile(0.05) if col == 'delay_days_int' else 0
    plot_data = df2[(df2[col] >= lower_limit) & (df2[col] <= upper_limit)]
    
    # 일수별 평균 리뷰 점수 시각화 (신뢰구간 포함)
    sns.lineplot(data=plot_data, x=col, y='review_score', linewidth=2.5,
                 ax=axes[i], color=info['color'], marker='o', markersize=3, errorbar=('ci', 95))
    
    # 타이틀 및 축 라벨 설정 (수식 포함)
    axes[i].set_title(f"{col.replace('_', ' ').title()}에 따른 리뷰스코어 추이", fontsize=12, fontweight='bold', pad=15)
    axes[i].set_xlabel(info['label'], fontsize=11, fontweight='semibold')
    axes[i].set_ylabel("평균 리뷰 스코어", fontsize=11)
    
    # 0점 기준선 (delivery_error_days의 경우 지연 여부 판단 기준)
    if col == 'delay_days_int':
        axes[i].axvline(0, color='gray', linestyle='--', alpha=0.5)
        axes[i].text(0.5, 4.5, 'On Time', color='gray', fontsize=10)

plt.tight_layout()
plt.show()


확실히 배송소요일이 오래 걸리거나 지연이 발생하면 리뷰스코어에 크게 타격이 있는 것으로 보여짐

(delivery_days & delay_days 그래프 확인)

따라서 상관관계를 파악해보기로 함


In [ ]:
# 배송 소요 일수 포함 주요 일수 컬럼과 리뷰 스코어의 상관관계

# 피어슨은 선형관계를 측정하는데 현 데이터는 선형성을 측정하는 것이 적절하지 않음
# '경향성'을 파악하고 범주형(순서형) 데이터의 rank를 측정할 땐 스피어만이 더 적절함

# 1. 상관관계 분석 대상에서 타겟 변수 분리
features = ['delivery_days', 'dispatch_days', 'approved_days', 'expected_delivery_days', 'delay_days']
target = 'review_score'

div_cmap = mcolors.LinearSegmentedColormap.from_list(
    "corr_diverging", ["#A85227", "#F5F5F5", "#3B5987"]
)

for day in features:
    valid_df = df2[[day, target]]
    
    if not valid_df.empty:
        # Spearman 산출 및 P-value 검정
        corr, p_value = stats.spearmanr(valid_df[day], valid_df[target])
        
        print(f"[{day}]")
        print(f"- 스피어만 상관계수: {corr:.3f}")
        print(f"- 유의성(P-value): {p_value:.4e}")
        
        # 해석 가이드 추가
        strength = "강한" if abs(corr) > 0.5 else "중간" if abs(corr) > 0.3 else "약한"
        sign = "음" if corr < 0 else "양" 
        print(f"- 해석: {strength} 정도의 {sign}의 상관관계\n")

# 3. 히트맵 시각화
plt.figure(figsize=(12, 10))

plt.figure(figsize=(12, 8))
sns.heatmap(df2[features + [target]].corr(method='spearman'), 
            annot=True, cmap='RdBu_r', center=0, fmt=".2f")
plt.title("물류 지표와 리뷰 점수 간 상관관계 (Spearman)")
plt.show()


[delivery_days] -0.222

[dispatch_days] -0.106

[approved_days] -0.020

[expected_delivery_days] -0.064

[delay_days] -0.152

- 배송 소요일(delivery_days)은 review_score와 -0.222의 약한 음의 상관관계를 보였다.
- 지연일, 출고 소요일보다 배송소요일의 단변량 상관계수가 상대적으로 크게 관찰되었다.
- 다만 상관분석은 지역, 거리, 카테고리, 판매자 특성을 통제하지 않으므로 인과나 독립적인 영향력으로 해석하면 안 된다. 여기서는 “배송소요일이 리뷰점수와 더 강하게 함께 움직이는 후보 변수”로 보는 것이 적절하다.


위에서 지연이 얼마나 되었는지보다 배송소요일이 review score에 더 강한 음의 상관관계를 보인다고 했는데 

그렇다면 지연이 되었는지의 여부 자체가 review score에 부정적인 영향을 끼치는지 확인해보고자 하였다.


### 정규성 검정 (qq-plot + shapiro 검정) --> welch's vs mannwhitney u 


In [ ]:
# 그룹 나누기 (지연 O vs 지연 X)
# 리뷰 점수는 주문 단위이므로, 배송 지연 여부 검정도 order_id 기준 주문 단위로 축약한다.
order_level_review = (
    df2[['order_id', 'is_delayed', 'review_score']]
    .dropna(subset=['order_id', 'is_delayed', 'review_score'])
    .drop_duplicates(subset=['order_id'])
    .copy()
)

delayed = order_level_review.loc[order_level_review['is_delayed'] == 1, 'review_score']
not_delayed = order_level_review.loc[order_level_review['is_delayed'] == 0, 'review_score']

# 두 그룹의 정규성 검정을 위한 q-q plot 시각화
fig, ax = plt.subplots(1,2, figsize=(12,4))
stats.probplot(delayed, plot=ax[0])
ax[0].set_title("지연 배송")
stats.probplot(not_delayed, plot=ax[1])
ax[1].set_title("정상/조기 배송")
plt.show()


In [ ]:
# shapiro 검정
stat_d, p_d = stats.shapiro(delayed)
stat_nd, p_nd = stats.shapiro(not_delayed)
print(f"지연 배송의 p-value : {p_d}")
print(f"정상/조기 배송의 p-value : {p_nd}")

print("두 그룹 모두 p-value가 0.05 미만이므로 정규성은 기각된다")
print("다만 표본 수가 매우 크고 review_score가 1~5점 순서형 평점이므로, 데이터 특성상 Mann-Whitney U 검정을 사용하는 것이 적절하다")


In [ ]:
print("=== 배송 지연에 따른 리뷰 스코어 분석 결과 ===")
print(f"정상/조기 배송 (0) 평균 점수: {not_delayed.mean():.2f}점 (데이터: {len(not_delayed):,}건)")
print(f"지연 배송 (1) 평균 점수: {delayed.mean():.2f}점 (데이터: {len(delayed):,}건)")
print(f"두 그룹의 평균 점수 차이: {not_delayed.mean() - delayed.mean():.2f}점")

# 지연 여부에 따른 평균 리뷰 점수 차이 확인 (Mann-Whitney U test)

u_stat, p_val = stats.mannwhitneyu(delayed, not_delayed)
print(f"\n집단 간 차이 유의성 (p-value): {p_val:.4e}")
if p_val < 0.05:
    print("=> 결론: P-value가 0.05보다 작으므로, 배송 지연 여부에 따른 리뷰 점수 차이는 통계적으로 유의미하다")
else:
    print("=> 결론: 통계적으로 유의미한 차이가 없다")

# --- 시각화 (컬러 변경 버전) ---
# 1. 커스텀 팔레트 정의 (0: 블루, 1: 오렌지)
my_palette = {0: "#4C72B0", 1: "#DD8452"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6)) # 가로 크기를 살짝 키워 여유 배치

# [왼쪽] 평균 점수 Barplot
# palette 매개변수에 커스텀 팔레트 딕셔너리를 직접 전달합니다.
sns.barplot(
    data=order_level_review, 
    x='is_delayed', 
    y='review_score', 
    ax=axes[0], 
    palette=my_palette,
    hue='is_delayed',
    edgecolor='gray',
    alpha=0.8
)

axes[0].set_title('배송 지연 여부에 따른 평균 리뷰 점수 비교', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['정상/조기 배송 (0)', '지연 배송 (1)'], fontsize=11)
axes[0].set_ylabel('평균 리뷰 스코어', fontsize=12)

# [오른쪽] 점수 분포 Boxplot
sns.boxplot(
    data=order_level_review, 
    x='is_delayed', 
    y='review_score', 
    ax=axes[1], 
    palette=my_palette,
    hue='is_delayed',
    linewidth=1.5,
    boxprops={'alpha': 0.8},
    fliersize=3 # 아웃라이어 크기 조절
)

axes[1].set_title('배송 지연 여부에 따른 리뷰 점수 분포 상세', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['정상/조기 배송 (0)', '지연 배송 (1)'], fontsize=11)
axes[1].set_ylabel('리뷰 스코어 (1~5)', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# 비모수 효과크기 rank-biserial correlation
r_rb = pg.mwu(delayed, not_delayed)
display(r_rb)


# 효과크기 해석 기준 반영
def interpret_rbc(rbc):
    abs_rbc = abs(rbc)
    if abs_rbc < 0.1: return "심지어 무시할 만한 수준 (Negligible)"
    elif abs_rbc < 0.3: return "작은 수준 (Small)"
    elif abs_rbc < 0.5: return "중간 수준 (Medium)"
    else: return "큰 수준 (Large)"

print(f"RBC 기반 효과 크기: {r_rb['RBC'].values[0]:.3f} -> {interpret_rbc(r_rb['RBC'].values[0])}")


### 지연 유무에 따른 리뷰스코어 차이 통계검정 결과
- Mann-Whitney U 검정 결과, 배송 지연 그룹과 정상/조기 배송 그룹 간 리뷰 점수 분포 차이는 통계적으로 매우 유의미하였다 (p < .001).
- 평균 점수 차이는 1.68점이며, 효과크기(RBC)는 -0.54로 큰 수준으로 나타났다.
- 리뷰점수는 1점부터 5점까지의 이산형 평점이므로, 정규성 검정 결과보다 순서형 평점이라는 데이터 특성을 근거로 비모수 검정을 사용하는 것이 더 적절하다.
- 따라서 단순 배송 소요 시간을 줄이는 것도 중요하지만, 약속된 기한 내 배송 여부를 관리하는 것이 리뷰 점수 하락을 방어하는 핵심 KPI 후보가 될 수 있다.


In [ ]:
# 1. 온타임 그룹 정의 (delay_days <= 0)
df_ontime = df2[df2['delay_days'] <= 0].copy()

# 2. 온타임 그룹 내에서 배송 소요 일수의 중앙값 산출
# 중앙값을 기준으로 하면 데이터의 치우침(Skewness)에 강건한 기준을 세울 수 있습니다.
ontime_duration_median = df_ontime['delivery_days'].median()
print(f"온타임 그룹의 배송 소요일 중앙값: {ontime_duration_median:.1f}일")

# 3. 세 세부 그룹 생성
# 그룹 1: 정시에 도착했고, 절대적 속도도 빠른 경우
ontime_fast = df_ontime[df_ontime['delivery_days'] <= ontime_duration_median]['review_score']

# 그룹 2: 정시에 도착했으나, 절대적 속도는 느린 경우 (약속은 지켰지만 오래 기다림)
ontime_slow = df_ontime[df_ontime['delivery_days'] > ontime_duration_median]['review_score']

# 그룹 3: 약속된 날짜보다 늦게 도착한 경우 (지연)
delayed_group = df2[df2['delay_days'] > 0]['review_score']

# 4. 결과 출력
print("-" * 30)
print(f"1. 정시-신속 그룹 평균 점수: {ontime_fast.mean():.2f} (n={len(ontime_fast)})")
print(f"2. 정시-저속 그룹 평균 점수: {ontime_slow.mean():.2f} (n={len(ontime_slow)})")
print(f"3. 배송 지연 그룹 평균 점수: {delayed_group.mean():.2f} (n={len(delayed_group)})")

# 5. 통계적 유의성 확인 (Kruskal-Wallis 검정: 세 집단 이상의 평균 차이 확인)
from scipy.stats import kruskal
stat, p_val = kruskal(ontime_fast, ontime_slow, delayed_group)
print(f"\n세 집단 간 차이 유의성 (p-value): {p_val:.4e}")


In [ ]:
4.11 / 2.54


정시에 배송완료가 되었더라도 배송소요일의 중앙값보다 늦은 경우라면 리뷰 스코어가 낮아지는지 확인하기 위해 세 그룹으로 나누어 비교하였다.

결과적으로 정시-저속 그룹의 평균 리뷰점수는 4.10점으로 정시-신속 그룹의 4.31점보다 낮았다.

다만 배송 지연 그룹의 평균 2.55점보다는 1.55점 높았다. 리뷰점수는 비율척도가 아니므로 “몇 배 높다”가 아니라 “몇 점 높다”로 해석하는 것이 정확하다.


In [ ]:
# 1. 시각화 데이터 준비 (이상치 제거 - 상위 95% 수준)
upper_limit = df2['delivery_days'].quantile(0.95)
plot_data = df2[df2['delivery_days'] <= upper_limit].copy()

# 2. 범례(Legend)를 위한 지연 여부 라벨링
plot_data['status'] = plot_data['is_delayed'].map({1: 'Delayed', 0: 'On-time'})

# 3. 시각화
plt.figure(figsize=(12, 8))
sns.lineplot(data=plot_data, x='delivery_days', y='review_score', 
             hue='status', palette=['#4C72B0', "#DD8452"], 
             marker='o', linewidth=3, markersize=4, errorbar=('ci', 95))

# 4. 그래프 디테일 설정
plt.title("배송소요일에 따른 평균 리뷰스코어 변화", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("배송소요일", fontsize=12)
plt.ylabel("평균 리뷰스코어", fontsize=12)
plt.legend(title="Delivery Status", frameon=True)

# 5. 인사이트 강화를 위한 수직선 (예: 전체 평균 소요일)
plt.axvline(df2['delivery_days'].median(), color='gray', linestyle='--', alpha=0.5)
plt.text(df2['delivery_days'].median() + 0.5, 4.5, 'Median Duration', color='gray', fontsize=10)

plt.tight_layout()
plt.show()


지연된 그룹과 정시에 도착한 그룹을 나누어 실제 배송소요일에 따른 리뷰스코어 변화를 확인하였다.

일부 지연 그룹은 실제 배송소요일이 전체 중앙값보다 짧더라도 정시/조기 배송 그룹보다 리뷰점수가 낮게 관찰되었다. 이는 고객이 절대적인 배송 속도뿐 아니라 약속된 예상도착일 준수 여부에도 민감할 수 있다는 가설을 뒷받침한다.

따라서 예상도착일(estimated)을 더 보수적으로 산정하는 전략은 검토할 만한 가설이다. 다만 구매 전환율, 지역, 거리, 카테고리, 판매자 처리속도를 통제하지 않았으므로 이 단계에서는 정책 추천이 아니라 추가 검증이 필요한 분석 가설로 둔다.


In [ ]:
# 예상배송소요일(expected_delivery_days)에 따른 리뷰스코어 변화 추이 시각화
upper_limit = df2['expected_delivery_days'].quantile(0.95) # 이상치 제거 - 상위 95% 수준
plot_data = df2[df2['expected_delivery_days'] <= upper_limit].copy()

plt.figure(figsize=(12, 8))
sns.lineplot(data=plot_data, x='expected_delivery_days', y='review_score', linewidth=3, 
             marker='o', markersize=4, color='#4C72B0', errorbar=('ci', 95))

plt.title("예상배송소요일에 따른 리뷰스코어 변화", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Expected delivery days", fontsize=12)
plt.ylabel("Average Review Score", fontsize=12)
plt.yticks(fontsize=14)  # y축 숫자 크기 키우기
plt.legend(title="Delivery Status", frameon=True)
plt.grid(True, alpha=0.5)

plt.tight_layout()
plt.show()


예상배송소요일(expected_delivery_days)과 리뷰스코어의 스피어만 상관계수는 -0.064로 매우 약한 수준이다.

따라서 현재 데이터에서는 예상배송소요일이 길수록 리뷰점수가 크게 낮아진다는 증거는 강하지 않다.

다만 이 결과만으로 “예상배송일을 길게 잡는 전략이 효과적”이라고 결론 내릴 수는 없다. 실제 정책 판단을 위해서는 예상배송일 변경이 구매 전환율, 지연율, 리뷰점수에 미치는 영향을 함께 검증해야 한다.


In [ ]:
# 배송 지연 구간별 평점 하락폭 분석 (Bar)
# delay_days: 양수면 지연 도착, 0이면 정시, 음수면 조기 도착

# 1. 구간 순서와 색상을 1:1로 매칭하는 딕셔너리 생성
order = ['조기', '정시', '1-3일 지연', '4-6일 지연', '7-9일 지연', '10일 이상 지연']
my_palette = {
    '조기': "#4C72B0",           # 블루
    '정시': "#7091C2",           # 연한 블루
    '1-3일 지연': "#E1A679",      # 연한 오렌지
    '4-6일 지연': "#DD8452",      # 오렌지
    '7-9일 지연': "#C36A3B",      # 진한 오렌지
    '10일 이상 지연': "#A85227"    # 레드/브라운
}

plt.figure(figsize=(12, 8))

# 2. barplot 그리기
sns.barplot(
    data=df2, 
    x='delay_days_cat', 
    y='review_score', 
    order=order, 
    palette=my_palette,     
    hue='delay_days_cat',   
    legend=False,            
    edgecolor='gray',
    alpha=0.8
)

# 3. 그래프 디테일
plt.title('배송 지연 구간에 따른 평균 리뷰 점수 하락폭', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('배송 지연 구간', fontsize=12)
plt.ylabel('평균 리뷰 점수 (1~5점)', fontsize=12)
plt.ylim(1, 5)

# 값 표시 추가
ax = plt.gca()
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 9), 
                textcoords='offset points',
                fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# 포트폴리오용 3일 초과 지연 기준 차트 저장
# 1-3일 지연에서 4일 이상 지연으로 넘어갈 때 리뷰 점수가 급락하는 패턴을 이미지 파일로 남긴다.
threshold_chart_order = ['조기', '정시', '1-3일 지연', '4-6일 지연', '7-9일 지연', '10일 이상 지연']
threshold_chart_palette = {
    '조기': '#D8E2F0',
    '정시': '#B8C7DA',
    '1-3일 지연': '#8FA6C1',
    '4-6일 지연': '#003478',
    '7-9일 지연': '#002A61',
    '10일 이상 지연': '#001F49'
}

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(
    data=df2,
    x='delay_days_cat',
    y='review_score',
    order=threshold_chart_order,
    palette=threshold_chart_palette,
    hue='delay_days_cat',
    legend=False,
    edgecolor='#D7DEE8',
    alpha=0.95,
    ax=ax
)

ax.axvline(2.5, color='#B00020', linestyle='--', linewidth=1.5)
ax.text(2.56, 4.75, '3일 초과 기준', color='#B00020', fontsize=11, fontweight='bold')
ax.set_title('배송 지연 구간에 따른 평균 리뷰 점수 하락폭', fontsize=15, fontweight='bold', pad=18)
ax.set_xlabel('배송 지연 구간', fontsize=11)
ax.set_ylabel('평균 리뷰 점수 (1-5점)', fontsize=11)
ax.set_ylim(1, 5)

for patch in ax.patches:
    height = patch.get_height()
    ax.annotate(
        f'{height:.2f}',
        (patch.get_x() + patch.get_width() / 2, height),
        ha='center',
        va='bottom',
        xytext=(0, 6),
        textcoords='offset points',
        fontsize=10,
        fontweight='bold'
    )

ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
chart_path = CHART_DIR / 'delay_review_threshold_bar.png'
fig.savefig(chart_path, dpi=220, bbox_inches='tight')
print(f'차트 저장 완료: {chart_path}')
plt.show()


In [ ]:
df2['delay_days_cat'].value_counts()


In [ ]:
# 포트폴리오용 핵심 차트 저장
# 포트폴리오에 들어가는 리뷰 점수 주장은 order_id 기준 주문 단위로 맞춘다.
portfolio_chart_files = []

sm_blue = '#003478'
sm_blue_dark = '#001F49'
sm_blue_mid = '#4F6F9D'
sm_blue_light = '#D8E2F0'
sm_line = '#D7DEE8'
sm_red = '#B00020'

portfolio_order_level = (
    df2[['order_id', 'is_delayed', 'review_score', 'delay_days', 'delay_days_cat', 'dispatch_days']]
    .dropna(subset=['order_id', 'is_delayed', 'review_score'])
    .drop_duplicates(subset=['order_id'])
    .copy()
)

# 1) 배송 지연 여부에 따른 평균 리뷰 점수 비교
fig, ax = plt.subplots(figsize=(8, 5.2))
delay_review_summary = (
    portfolio_order_level.groupby('is_delayed')['review_score']
    .mean()
    .reindex([0, 1])
)
labels = ['정상·조기 배송', '지연 배송']
colors = [sm_blue_light, sm_blue]
bars = ax.bar(labels, delay_review_summary.values, color=colors, edgecolor=sm_line, linewidth=1.2)
ax.set_title('배송 지연 여부에 따른 평균 리뷰 점수 차이', fontsize=15, fontweight='bold', pad=18)
ax.set_ylabel('평균 리뷰 점수 (1-5점)', fontsize=11)
ax.set_ylim(1, 5)
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', (bar.get_x() + bar.get_width() / 2, height),
                ha='center', va='bottom', xytext=(0, 7), textcoords='offset points',
                fontsize=12, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', color='#EEF2F7', linewidth=1)
ax.set_axisbelow(True)
plt.tight_layout()
chart_path = CHART_DIR / 'portfolio_delay_review_gap.png'
fig.savefig(chart_path, dpi=220, bbox_inches='tight')
portfolio_chart_files.append(chart_path)
plt.show()

# 2) 3일 초과 지연 기준 차트
threshold_plot_data = portfolio_order_level.dropna(subset=['delay_days']).copy()
threshold_plot_data['delay_threshold_chart_group'] = np.select(
    [
        threshold_plot_data['delay_days'] < 0,
        threshold_plot_data['delay_days'] == 0,
        (threshold_plot_data['delay_days'] > 0) & (threshold_plot_data['delay_days'] <= 3),
        threshold_plot_data['delay_days'] > 3,
    ],
    ['조기', '정시', '1-3일 지연', '4일 이상 지연'],
    default='기타'
)
threshold_chart_order = ['조기', '정시', '1-3일 지연', '4일 이상 지연']
threshold_chart_palette = ['#E8F0FA', '#BFD0E6', '#7FA0C8', sm_blue]
threshold_review_summary = (
    threshold_plot_data.groupby('delay_threshold_chart_group')['review_score']
    .mean()
    .reindex(threshold_chart_order)
)
fig, ax = plt.subplots(figsize=(9, 5.6))
bars = ax.bar(
    threshold_chart_order,
    threshold_review_summary.values,
    color=threshold_chart_palette,
    edgecolor=sm_line,
    linewidth=1.2,
    alpha=0.98,
)
ax.axvline(2.5, color=sm_red, linestyle='--', linewidth=1.5)
ax.text(2.56, 4.75, '3일 초과 기준', color=sm_red, fontsize=11, fontweight='bold')
ax.set_title('3일 초과 지연 기준 평균 리뷰 점수 차이', fontsize=15, fontweight='bold', pad=18)
ax.set_xlabel('배송 지연 구간', fontsize=11)
ax.set_ylabel('평균 리뷰 점수 (1-5점)', fontsize=11)
ax.set_ylim(1, 5)
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', (bar.get_x() + bar.get_width() / 2, height),
                ha='center', va='bottom', xytext=(0, 6), textcoords='offset points',
                fontsize=10, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', color='#EEF2F7', linewidth=1)
ax.set_axisbelow(True)
plt.tight_layout()
chart_path = CHART_DIR / 'delay_review_threshold_bar.png'
fig.savefig(chart_path, dpi=220, bbox_inches='tight')
portfolio_chart_files.append(chart_path)
plt.show()

# 3) 판매자 출고 속도에 따른 배송 지연 발생률
bins = [-np.inf, 1, 3, 5, np.inf]
dispatch_labels = ['당일-1일', '2-3일', '4-5일', '6일 이상']
dispatch_data = portfolio_order_level.dropna(subset=['dispatch_days']).copy()
dispatch_data['dispatch_speed_portfolio'] = pd.cut(dispatch_data['dispatch_days'], bins=bins, labels=dispatch_labels)
delay_rate = dispatch_data.groupby('dispatch_speed_portfolio', observed=False)['is_delayed'].mean().reindex(dispatch_labels) * 100

fig, ax = plt.subplots(figsize=(8.8, 5.2))
colors = [sm_blue_light, '#B8C7DA', sm_blue_mid, sm_blue]
bars = ax.bar(delay_rate.index.astype(str), delay_rate.values, color=colors, edgecolor=sm_line, linewidth=1.2)
ax.set_title('판매자 출고 속도에 따른 배송 지연 발생률', fontsize=15, fontweight='bold', pad=18)
ax.set_xlabel('판매자 출고 소요일', fontsize=11)
ax.set_ylabel('배송 지연 발생률 (%)', fontsize=11)
ax.set_ylim(0, max(delay_rate.values) + 12)
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', (bar.get_x() + bar.get_width() / 2, height),
                ha='center', va='bottom', xytext=(0, 7), textcoords='offset points',
                fontsize=11, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', color='#EEF2F7', linewidth=1)
ax.set_axisbelow(True)
plt.tight_layout()
chart_path = CHART_DIR / 'portfolio_dispatch_delay_rate.png'
fig.savefig(chart_path, dpi=220, bbox_inches='tight')
portfolio_chart_files.append(chart_path)
plt.show()

print('포트폴리오용 차트 저장 완료')
for chart_file in portfolio_chart_files:
    print(chart_file)


### 3일 초과 지연 기준 리뷰 점수 차이 검정

위 시각화에서 지연 구간이 1–3일에서 4일 이상으로 넘어갈 때 평균 리뷰 점수가 크게 낮아지는 패턴이 확인되었다.

다만 현재 분석 데이터는 주문-아이템 단위이고, 리뷰 점수는 주문 단위로 붙어 있다. 같은 주문의 리뷰가 여러 아이템 행에 반복될 수 있으므로, 3일 기준 검정은 `order_id` 기준으로 주문 1건당 1행만 남긴 뒤 수행한다.

검정 질문은 다음과 같다.

- 귀무가설: 1–3일 지연 주문과 4일 이상 지연 주문의 리뷰 점수 분포는 차이가 없다.
- 대립가설: 1–3일 지연 주문과 4일 이상 지연 주문의 리뷰 점수 분포는 차이가 있다.

`review_score`는 1점부터 5점까지의 순서형 평점이고 두 집단의 분포가 정규분포라고 보기 어려우므로 Mann-Whitney U 검정을 사용한다. p값만으로 판단하지 않기 위해 Rank-biserial correlation을 함께 계산한다.


In [ ]:
# 3일 초과 지연 기준 리뷰 점수 차이 검정
# 리뷰 점수는 order_id 단위이므로, 주문-아이템 중복을 피하기 위해 주문 단위로 축약한다.
order_level = (
    df2[['order_id', 'review_score', 'delay_days']]
    .dropna(subset=['order_id', 'review_score', 'delay_days'])
    .drop_duplicates(subset=['order_id'])
    .copy()
)

# 지연 주문만 대상으로 1-3일 지연과 4일 이상 지연을 비교한다.
delayed_orders = order_level[order_level['delay_days'] > 0].copy()
delayed_orders['delay_threshold_group'] = np.where(
    delayed_orders['delay_days'] <= 3,
    '1-3일 지연',
    '4일 이상 지연'
)

threshold_summary = (
    delayed_orders
    .groupby('delay_threshold_group')['review_score']
    .agg(['count', 'mean', 'median', 'std'])
    .reindex(['1-3일 지연', '4일 이상 지연'])
)

short_delay_scores = delayed_orders.loc[
    delayed_orders['delay_threshold_group'] == '1-3일 지연',
    'review_score'
]
long_delay_scores = delayed_orders.loc[
    delayed_orders['delay_threshold_group'] == '4일 이상 지연',
    'review_score'
]

u_stat, p_value = stats.mannwhitneyu(
    short_delay_scores,
    long_delay_scores,
    alternative='two-sided'
)

# Rank-biserial correlation: 양수면 1-3일 지연 그룹의 리뷰 점수가 4일 이상 지연 그룹보다 높은 방향이다.
rank_biserial_r = (2 * u_stat / (len(short_delay_scores) * len(long_delay_scores))) - 1
mean_diff = short_delay_scores.mean() - long_delay_scores.mean()

print('=== 3일 초과 지연 기준 리뷰 점수 차이 검정 ===')
display(threshold_summary)
print(f"평균 리뷰 점수 차이(1-3일 지연 - 4일 이상 지연): {mean_diff:.2f}점")
print(f"Mann-Whitney U statistic: {u_stat:.0f}")
print(f"p-value: {p_value:.4e}")
print(f"Rank-biserial correlation: {rank_biserial_r:.3f}")

if abs(rank_biserial_r) >= 0.5:
    effect_label = '큰 효과'
elif abs(rank_biserial_r) >= 0.3:
    effect_label = '중간 효과'
elif abs(rank_biserial_r) >= 0.1:
    effect_label = '작은 효과'
else:
    effect_label = '매우 작은 효과'

print(f"효과크기 해석: {effect_label}")
print('해석: 4일 이상 지연 주문은 1-3일 지연 주문보다 리뷰 점수가 유의하게 낮으며, 효과크기도 실질적으로 큰 수준이다.')

threshold_result = threshold_summary.reset_index().rename(columns={'delay_threshold_group': 'group'})
threshold_result['mean_diff_1_3_minus_4_plus'] = mean_diff
threshold_result['mannwhitney_u'] = u_stat
threshold_result['p_value'] = p_value
threshold_result['rank_biserial_r'] = rank_biserial_r
threshold_result['effect_size_label'] = effect_label
threshold_result.to_csv(TABLE_DIR / 'delay_threshold_test_summary.csv', index=False)


In [ ]:
delayed_dispatch = df2[df2['is_delayed'] == 1]['dispatch_days']
not_delayed_dispatch = df2[df2['is_delayed'] == 0]['dispatch_days']

stat_d, p_d = stats.shapiro(delayed_dispatch)
stat_nd, p_nd = stats.shapiro(not_delayed_dispatch)
print(f"delayed_dispatch의 p-value: {p_d}")
print(f"not_delayed_dispatch의 p-value: {p_nd}")
print("두 집단 모두 p-value가 0.05 미만이므로 정규성은 기각된다")
print("표본 수가 크고 분포가 치우쳐 있으므로 평균 비교보다 Mann-Whitney U 검정으로 순위 기반 차이를 확인한다")


In [ ]:
u_stat, u_p = stats.mannwhitneyu(delayed_dispatch, not_delayed_dispatch, alternative="greater")
print(f"맨휘트니-U검정 결과 p-value: {u_p}")  # 0.0
print("p-value가 0.05 미만이므로 지연배송의 출고소요일이 정상/조기배송보다 크다")


In [ ]:
# 비모수 효과크기 rank-biserial correlation
r_rb = pg.mwu(delayed_dispatch, not_delayed_dispatch)
display(r_rb)

print(r_rb['RBC'].values[0]) # 효과크기가 중간이다!
print("RBC 효과크기가 0.3 이상으로 중간정도이다")
print("CLES가 0.65라는 것은 지연배송의 출고소요일이 정상/조기배송보다 길 확률이 65%라는 것!")


In [ ]:
print("=== 지연 여부에 따른 판매자 출고 시간 분석 ===")
print(f"정상/조기 배송 건의 평균 출고 시간: {not_delayed_dispatch.mean():.2f}일")
print(f"지연 배송 건의 평균 출고 시간: {delayed_dispatch.mean():.2f}일")
print(f"차이: 지연된 주문은 평균적으로 {delayed_dispatch.mean() - not_delayed_dispatch.mean():.2f}일 더 늦게 출고됨")

# 3. 출고 소요일 구간(카테고리)화 하여 지연 발생 확률(Delay Rate) 계산
# 구간 설정: 0~1일(빠름), 2~3일(보통), 4~5일(느림), 6일 이상(매우 느림)
bins = [-np.inf, 1, 3, 5, np.inf]
labels = ['당일~1일 (빠름)', '2~3일 (보통)', '4~5일 (느림)', '6일 이상 (매우 느림)']
df2['dispatch_speed'] = pd.cut(df2['dispatch_days'], bins=bins, labels=labels)

# 구간별 지연율 계산 (지연된 건수 / 전체 건수 * 100)
delay_rate = df2.groupby('dispatch_speed')['is_delayed'].mean() * 100

plt.rc('font', family='AppleGothic') 
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 컬러 팔레트 정의
# 0: 블루(정상), 1: 오렌지(지연)
status_palette = {0: "#4C72B0", 1: "#DD8452"}
# 지연율 팔레트: 빠름(연한 블루) -> 매우 느림(짙은 오렌지/레드)
rate_palette = ["#7091C2", "#B9C0C9", "#DD8452", "#C36A3B"]

# [왼쪽] 배송 지연 여부에 따른 출고 소요일 분포 (Boxplot)
sns.boxplot(
    data=df2, 
    x='is_delayed', 
    y='dispatch_days', 
    ax=axes[0], 
    palette=status_palette,
    hue='is_delayed',
    legend=False,
    boxprops={'alpha': 0.8}, 
    linewidth=1.5,
    fliersize=3
)
axes[0].set_title('배송 지연 여부에 따른 출고 소요일 분포', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['정상/조기 배송 (0)', '지연 배송 (1)'])
axes[0].set_ylabel('출고 소요일 (Dispatch Days)')
axes[0].set_ylim(-1, 15) 

# [오른쪽] 출고 속도 구간별 배송 지연 발생률 (Barplot)
sns.barplot(
    x=delay_rate.index, 
    y=delay_rate.values, 
    ax=axes[1], 
    palette=rate_palette,
    hue=delay_rate.index,
    dodge=False,
    legend=False,
    edgecolor='gray',
    alpha=0.85
)
axes[1].set_title('판매자 출고 속도에 따른 배송 지연 발생률(%)', fontsize=14, fontweight='bold', pad=15)
axes[1].set_ylabel('지연 발생 확률 (%)')
axes[1].set_xlabel('판매자 출고 속도')
axes[1].set_ylim(0, max(delay_rate.values) + 15) # 텍스트 공간 확보

# 막대 그래프 위에 퍼센트(%) 텍스트 추가
for i, v in enumerate(delay_rate.values):
    axes[1].text(i, v + 1.5, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
df2['dispatch_speed'].value_counts()


In [ ]:
# 출고 기간(0~10일)별 평균 리뷰 점수 시각화
plt.figure(figsize=(12, 8))
dispatch_perf = df2[df2['dispatch_days'].between(0, 10)].groupby('dispatch_days')['review_score'].mean()

plt.plot(dispatch_perf.index, dispatch_perf.values, marker='o', color='#4C72B0', linewidth=2)
plt.title('판매자 출고 준비 기간과 고객 만족도 관계', fontsize=14)
plt.xlabel('출고 준비 기간 (일)', fontsize=12)
plt.ylabel('평균 리뷰 점수', fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# 데이터 준비
# 판매자 vs 택배사 책임 소재 분석
# 전체 배송일(delivery_days) = 출고 소요(dispatch_days) + 순수 배송(carrier_days)
df2['carrier_days'] = df2['delivery_days'] - df2['dispatch_days']

# 평점별 평균 소요 시간 비교
responsibility = df2.groupby(df2['review_score'].round().astype(int))[['dispatch_days', 'carrier_days']].mean()

# x축: 리뷰 점수 (1, 2, 3, 4, 5)
x = responsibility.index
y_dispatch = responsibility['dispatch_days']
y_carrier = responsibility['carrier_days']
y_total = y_dispatch + y_carrier  # 전체 높이 (택배사 막대의 끝점)

# 누적 막대 그래프 그리기 (Background)
# 판매자 준비 기간 (바닥부터 시작)
ax.bar(x, y_dispatch, color="#DD8452", alpha=0.8, label='판매자 준비(Dispatch)', edgecolor='white')
# 택배사 배송 기간 (판매자 막대 위부터 시작)
ax.bar(x, y_carrier, bottom=y_dispatch, color="#4C72B0", alpha=0.8, label='택배사 배송(Carrier)', edgecolor='white')

# 막대 끝점을 잇는 선 그리기 (Line Plot)
# 선 1: 판매자 준비 기간의 끝점 (첫 번째 세그먼트 상단)
ax.plot(x, y_dispatch, marker='o', linestyle='-', color='#A85227', 
        linewidth=2.5, markersize=8, label='판매자 준비 추이')

# 선 2: 전체 배송 완료 시점 (두 번째 세그먼트 상단 = 전체 높이)
ax.plot(x, y_total, marker='s', linestyle='-', color='#2B456B', 
        linewidth=2.5, markersize=8, label='전체 배송 완료 추이')

# 세부 스타일링
plt.title('리뷰 점수별 물류 단계별 소요 시간 및 추이 분석', fontsize=16, fontweight='bold', pad=25)
plt.xlabel('리뷰 점수', fontsize=12)
plt.ylabel('평균 소요 일수 (Days)', fontsize=12)

# 범례 설정
ax.legend(loc='upper right', frameon=True, fontsize=10)
plt.xticks(x)

plt.tight_layout()
plt.show()


In [ ]:
df2.columns


#### customer state 별로!


In [ ]:
# 1. 데이터 준비 및 전처리 (주문 건수가 너무 적은 노이즈 지역 필터링)
df_clean = df2.copy()

# 지역별 통계 계산
state_stats = df_clean.groupby('customer_state').agg(
    total_orders=('order_item_id', 'count'),
    delay_rate=('is_delayed', lambda x: x.mean() * 100),    # 지연율(%)
    avg_review_score=('review_score', 'mean')               # 평균 리뷰 점수
).reset_index()

display(state_stats)

# 데이터 신뢰성을 위해 주문 건수 100건 이상인 지역(State)만 분석
state_stats = state_stats[state_stats['total_orders'] > 100]

# 지연율이 높은 순서대로(내림차순) 정렬 
state_stats_sorted = state_stats.sort_values(by='delay_rate', ascending=False)

# 전체 평균 계산 (비교 기준선용)
overall_delay_rate = df_clean['is_delayed'].mean() * 100
overall_review_score = df_clean['review_score'].mean()

# 1. 그라데이션 팔레트 생성 (데이터 개수에 맞춰 설정)
n_states = len(state_stats_sorted)
orange_grad = sns.light_palette("#DD8452", n_colors=n_states+5, reverse=True)
blue_grad = sns.light_palette("#4C72B0", n_colors=n_states+5, reverse=False)

fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharex=True)

# [위쪽 그래프] 지역별 배송 지연율 (%)
sns.barplot(
    data=state_stats_sorted, 
    x='customer_state', 
    y='delay_rate', 
    ax=axes[0], 
    palette=orange_grad,
    hue='customer_state',
    legend=False,
    edgecolor='gray',      # 테두리 색상 설정 (검정이나 진한 회색 추천)
    linewidth=1.5          # 테두리 두께 설정
)
axes[0].set_title('지역(State)별 배송 지연 발생률 (%) - 위험 지역부터 정렬', fontsize=15, fontweight='bold', pad=15)
axes[0].axhline(overall_delay_rate, color='black', linestyle='--', linewidth=2, label='전체 평균')
axes[0].legend(loc='upper right')

# [아래쪽 그래프] 지역별 평균 리뷰 스코어
sns.barplot(
    data=state_stats_sorted, 
    x='customer_state', 
    y='avg_review_score', 
    ax=axes[1], 
    palette=blue_grad,
    order=state_stats_sorted['customer_state'],
    hue='customer_state',
    legend=False,
    edgecolor='gray',      # 테두리 색상 설정
    linewidth=1.5          # 테두리 두께 설정
)
axes[1].set_title('지역별 평균 리뷰 스코어 (지연 리스크와 평점의 관계 확인)', fontsize=15, fontweight='bold', pad=15)
axes[1].axhline(overall_review_score, color='black', linestyle='--', linewidth=2, label='전체 평균')
axes[1].set_ylim(3.5, 4.8) 
axes


In [ ]:
# 중분류별 지표 요약
sub_stats = df2.groupby('sub_category').agg({
    'delivery_days': 'mean',
    'is_delayed': 'mean',
    'review_score': 'mean'
}).rename(columns={'is_delayed': 'delay_rate'}).reset_index()

sub_stats['delay_rate'] = sub_stats['delay_rate'] * 100


In [ ]:
# 산점도 그리기
sns.scatterplot(
    data=sub_stats, 
    x='delivery_days', 
    y='delay_rate', 
    size='review_score',  # 리뷰 점수가 낮을수록 원이 작아지게 설정
    hue='sub_category', 
    legend=False,
    sizes=(100, 500),
    alpha=0.8
)

# 각 점에 카테고리 이름 표시
for i in range(sub_stats.shape[0]):
    plt.text(
        sub_stats.delivery_days[i] - 0.1, 
        sub_stats.delay_rate[i] + 0.2,  # 텍스트가 점과 겹치지 않도록 약간 위로 이동
        sub_stats.sub_category[i], 
        fontsize=9, alpha=0.8
    )

# 평균선(기준선) 추가 - 사분면 구분
plt.axvline(sub_stats['delivery_days'].mean(), color='gray', linestyle='--', alpha=0.5)
plt.axhline(sub_stats['delay_rate'].mean(), color='gray', linestyle='--', alpha=0.5)

plt.title('배송소요일과 지연률 산점도 - 사분면 시각화', fontsize=15)
plt.xlabel('평균 배송소요일 (Days)')
plt.ylabel('지연률 (Rate)')

# 우측 상단 텍스트 추가
plt.text(sub_stats['delivery_days'].max()*0.95, sub_stats['delay_rate'].max()*0.9, 
         'Critical Risk Area', color='red', fontweight='bold', fontsize=12)

plt.grid(True, linestyle=':', alpha=0.6)
plt.show()


In [ ]:
sub_avg_reviews = df2.groupby('sub_category')['review_score'].mean().sort_values(ascending=False).reset_index()

# 시각화 설정
plt.figure(figsize=(10, 12))
orange_grad = sns.light_palette("#DD8452", n_colors=n_states+5, reverse=True)
orange_grad_6 = sns.light_palette("#DD8452", n_colors=10, reverse=True)

# 막대 그래프 그리기
sns.barplot(
    x='review_score', 
    y='sub_category', 
    data=sub_avg_reviews, 
    palette=orange_grad,
    edgecolor='gray'
)

# 그래프 제목 및 라벨 설정
plt.title('서브 카테고리별 평균 리뷰스코어', fontsize=16, pad=20)
plt.xlabel('Average Review Score', fontsize=12)
plt.ylabel('Sub-Category', fontsize=12)

# X축 범위 설정
plt.xlim(0, 5)

# 각 막대 끝에 점수 표시
for index, row in sub_avg_reviews.iterrows():
    plt.text(row.review_score + 0.05, index, f'{row.review_score:.2f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

# 시각화 설정
plt.figure(figsize=(10, 5))

# 막대 그래프 그리기
sns.barplot(
    x='review_score', 
    y='sub_category', 
    data=sub_avg_reviews.tail(6), 
    palette=orange_grad_6,
    edgecolor='gray'
)

# 그래프 제목 및 라벨 설정
plt.title('서브 카테고리별 평균 리뷰스코어 (하위 6개 카테고리)', fontsize=16, pad=20)
plt.xlabel('평균 리뷰스코어', fontsize=12)
plt.ylabel('서브 카테고리', fontsize=12)

# X축 범위 설정
plt.xlim(0, 5)

# 각 막대 끝에 점수 표시
for index, (i, row) in enumerate(sub_avg_reviews.tail(6).iterrows()):
    plt.text(row.review_score + 0.05, index, f'{row.review_score:.2f}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


#### seller state 별로


In [ ]:
# 1. 데이터 준비 및 전처리 (주문 건수가 너무 적은 노이즈 지역 필터링)
df_clean = df2.copy()

# 지역별 통계 계산
state_stats = df_clean.groupby('seller_state').agg(
    total_orders=('order_item_id', 'count'),
    delay_rate=('is_delayed', lambda x: x.mean() * 100),    # 지연율(%)
    avg_review_score=('review_score', 'mean')               # 평균 리뷰 점수
).reset_index()

display(state_stats)

# 데이터 신뢰성을 위해 주문 건수 100건 이상인 지역(State)만 분석
state_stats = state_stats[state_stats['total_orders'] > 100]

# 지연율이 높은 순서대로(내림차순) 정렬 
state_stats_sorted = state_stats.sort_values(by='delay_rate', ascending=False)

# 전체 평균 계산 (비교 기준선용)
overall_delay_rate = df_clean['is_delayed'].mean() * 100
overall_review_score = df_clean['review_score'].mean()

# 1. 그라데이션 팔레트 생성 (데이터 개수에 맞춰 설정)
n_states = len(state_stats_sorted)
orange_grad = sns.light_palette("#DD8452", n_colors=n_states+5, reverse=True)
blue_grad = sns.light_palette("#4C72B0", n_colors=n_states+5, reverse=False)

fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharex=True)

# [위쪽 그래프] 지역별 배송 지연율 (%)
sns.barplot(
    data=state_stats_sorted, 
    x='seller_state', 
    y='delay_rate', 
    ax=axes[0], 
    palette=orange_grad,
    hue='seller_state',
    legend=False,
    edgecolor='gray',      # 테두리 색상 설정 (검정이나 진한 회색 추천)
    linewidth=1.5          # 테두리 두께 설정
)
axes[0].set_title('지역(State)별 배송 지연 발생률 (%) - 위험 지역부터 정렬', fontsize=15, fontweight='bold', pad=15)
axes[0].axhline(overall_delay_rate, color='black', linestyle='--', linewidth=2, label='전체 평균')
axes[0].legend(loc='upper right')

# [아래쪽 그래프] 지역별 평균 리뷰 스코어
sns.barplot(
    data=state_stats_sorted, 
    x='seller_state', 
    y='avg_review_score', 
    ax=axes[1], 
    palette=blue_grad,
    order=state_stats_sorted['seller_state'],
    hue='seller_state',
    legend=False,
    edgecolor='gray',      # 테두리 색상 설정
    linewidth=1.5          # 테두리 두께 설정
)
axes[1].set_title('지역별 평균 리뷰 스코어 (지연 리스크와 평점의 관계 확인)', fontsize=15, fontweight='bold', pad=15)
axes[1].axhline(overall_review_score, color='black', linestyle='--', linewidth=2, label='전체 평균')
axes[1].set_ylim(3.5, 4.8) 
axes


In [ ]:
# 판매자 state별 통계 계산
seller_state_stats = df2.groupby('seller_state').agg(
    total_orders=('order_item_id', 'count'),
    delay_rate=('is_delayed', lambda x: x.mean() * 100),    # 지연율(%)
    avg_review_score=('review_score', 'mean')               # 평균 리뷰 점수
).reset_index()


In [ ]:
# 판매자 state 별 카테고리
seller_state_cat = df2.groupby(['seller_state','main_category'])['order_item_id'].size().unstack(fill_value=0)
seller_state_cat.head(10)


In [ ]:
# 판매자 state별 카테고리 비율
seller_state_cat_ratio = seller_state_cat.div(seller_state_cat.sum(axis=1), axis=0).round(4) *100
seller_state_cat_ratio.head()


In [ ]:
# 상위 10개 state만

top_states = seller_state_cat.sum(axis=1).sort_values(ascending=False).head(10).index
top_state_ratio = seller_state_cat_ratio.loc[top_states].round(2)
top_state_ratio


In [ ]:
plt.figure(figsize=(12,10))

blue_cmap = mcolors.LinearSegmentedColormap.from_list("custom_blue", ["#F0F4FA", "#3B5987"])

sns.heatmap(
    top_state_ratio,
    annot=True,
    fmt='.2f',
    cmap=blue_cmap
)

plt.title('주 별 카테고리 비율 — 판매량 상위 10개 주')
plt.xlabel('카테고리 그룹')
plt.ylabel('판매자 거주 주')
plt.xticks(rotation=45, ha='right')
plt.show()


### 지연률이 높은 지역은 특정 카테고리 때문인가?
state x category x delay_rate


In [ ]:
# 지역별 카테고리별 지연률 
state_cat_summary = df2.groupby(['seller_state','main_category']).agg(
    delay_rate=('is_delayed', lambda x: x.mean()*100),
    orders=('order_item_id', 'count')
).reset_index()


In [ ]:
# seller_state별 판매량 정렬
state_order = state_cat_summary.groupby('seller_state')['orders'].sum().sort_values(ascending=False)

state_order.head()


In [ ]:
state_cat_delay = state_cat_summary.set_index(['seller_state', 'main_category'])['delay_rate'].unstack()
# 판매량 순으로 재정렬
state_cat_delay = state_cat_delay.loc[state_order.index]
top7_state_cat_delay = state_cat_delay.head(7)


In [ ]:
plt.figure(figsize=(10,8))

custom_cmap = mcolors.LinearSegmentedColormap.from_list("custom_orange", ["#F5F5F5", "#A85227"])

sns.heatmap(
    top7_state_cat_delay,
    cmap=custom_cmap,
    annot=True,
    fmt='.1f'
)

plt.title('주 X 카테고리별 배송지연률 — 판매량 상위 7개 주')
plt.xlabel('카테고리 그룹')
plt.ylabel('판매자 거주 주')
plt.xticks(rotation=45, ha='right')
plt.show()
